# Week 12: Assignment 4 — SMS Spam Detection

**Goal:** Apply the six-step classification workflow to a real dataset.
Predict whether an SMS message is spam or legitimate (ham) using text-based features.

---

## I. Assignment Objective

By completing this assignment, you will:
- Follow a complete classification pipeline on real-world data
- Engineer binary features from raw text
- Train and evaluate a Decision Tree and a Naive Bayes classifier
- Interpret a confusion matrix and choose the right evaluation metric
- Analyze model performance and reflect on possible improvements

---

## II. Dataset

**Source:** UCI SMS Spam Collection (5,572 real SMS messages, no login required)

| Column | Description | Example |
|---|---|---|
| `label` | Class label | `spam` or `ham` |
| `message` | Raw SMS text | "WINNER!! You have been selected..." |
| `spam` | Encoded label (1 = spam, 0 = ham) | 1 |

Load it with one line — no file upload needed:
```python
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
```

---

## III. Grading Rubric (Total = 70 points)

| Section | What is graded | Points |
|---|---|---|
| Step 1 — Problem definition | Q1a + Q1b + Q1c (written) | 5 |
| Step 2 — Explore the data | Code runs correctly (2B, 2C, 2D) | 10 |
| Step 3 — Feature engineering | Code (3B, 3C) + Q3a + Q3b + Q3c (written) | 15 |
| Step 4 — Split the data | Code (4A) + Q4a + Q4b (written) | 5 |
| Step 5 — Train models | Code: 5A + 5B + 5C + 5D (3 pts each) | 12|
| Step 6 — Code (6A + 6B + 6C) | Each code block runs correctly (4 pts each) | 12 |
| Step 6 — Written (Q6a–Q6d) | Q6a (2) + Q6b (3) + Q6c (3) + Q6d (3) | 11 |
| **Total** | | **70** |

---

## IV. Steps to Complete

Follow the steps below in order.
Each code cell starts with `# Write your code here` — replace it with your implementation.


---
### Setup — Run this cell first (provided)

In [ ]:
# This cell is provided — run it before starting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, ConfusionMatrixDisplay
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Setup complete!')


---
## Step 1 — Define the Problem

Before writing any code, answer the questions below.
Your answers here should guide every metric decision you make in Step 6.

**Context:**
A mobile carrier wants an automated classifier to flag spam SMS messages
before they reach users. The two possible errors are:

| Error | What happens |
|---|---|
| False Positive | A real (ham) message gets blocked |
| False Negative | A spam message reaches the user |

### Q1 — Problem Definition *(10 points)*

**Q1a.** Which error is more costly in a real spam filter — False Positive or False Negative?
Give one concrete real-world consequence for each.

*Your answer:*
> A **False Positive is more costly**. SMS is used for critical one-to-one messages, so blocking a legitimate one can cause real harm: for example, a bank's fraud alert or a two-factor authentication code gets blocked, and the user is locked out of their account or never learns that their card was used. A **False Negative** is mostly an annoyance: a "You won a £900 prize, call 0906…" message reaches the user, who can simply ignore or delete it. It is only dangerous if the user actually falls for it, and the user still has a chance to judge the message themselves, whereas a blocked message is invisible to them.

**Q1b.** Based on Q1a, which metric should be your primary evaluation target?
Choose one: **Precision**, **Recall**, or **F1-Score**. Explain why.

*Your answer:*
> **Precision.** Precision = TP / (TP + FP), i.e., of all the messages the model flags as spam, how many really are spam. Since False Positives (blocked real messages) are the expensive error, precision is the metric that directly penalizes them. A high-precision filter means that when it blocks a message, we can trust that it was spam. Recall should still be monitored so the filter is not useless, but it is the secondary target.

**Q1c.** The spam rate in this dataset is about 13%.
What accuracy would a dummy model that always predicts "ham" achieve?
Is that number still misleading? Why?

*Your answer:*
> The dummy model would reach about **87% accuracy** (86.6% in Step 2B), because 4,825 of the 5,572 messages are ham. This number is **very misleading**: the model catches **zero** spam messages (Recall = 0%, F1 = 0%). On an imbalanced dataset, accuracy is dominated by the majority class, so a model can look good while completely failing at the task we care about. This is why we evaluate with precision, recall and F1 instead of accuracy alone.


---
## Step 2 — Load and Explore the Data

### Step 2A — Load the dataset



In [ ]:
import urllib.request, zipfile, io

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
print('Downloading UCI SMS Spam Collection...')
with urllib.request.urlopen(url) as r:
    zf = zipfile.ZipFile(io.BytesIO(r.read()))
    with zf.open('SMSSpamCollection') as f:
        raw = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'])

raw['spam'] = (raw['label'] == 'spam').astype(int)
print(f'Loaded: {len(raw)} messages')
raw.head(6)


### Step 2B — Class distribution

**Hint:** Print the count and percentage of spam and ham messages.
Plot a bar chart with spam count in red and ham count in blue.
Also run a dummy model (predict all ham) and print its Accuracy, Recall, and F1.
This proves the Accuracy Trap from Lecture 8.


In [ ]:
# Write your code here
spam_count = raw['spam'].sum()
ham_count  = (raw['spam'] == 0).sum()
spam_rate  = raw['spam'].mean()

print('Class distribution:')
print(f'  Ham  (0): {ham_count:,}  ({1-spam_rate:.1%})')
print(f'  Spam (1): {spam_count:,}  ({spam_rate:.1%})')
print()

# Prove the accuracy trap with a dummy model
y_all   = raw['spam'].values
y_dummy = np.zeros(len(y_all), dtype=int)
print(f'Dummy model (always predict ham):')
print(f'  Accuracy  = {accuracy_score(y_all, y_dummy):.1%}  <-- misleadingly high')
print(f'  Recall    = {recall_score(y_all, y_dummy, zero_division=0):.1%}  <-- catches zero spam')
print(f'  F1-Score  = {f1_score(y_all, y_dummy, zero_division=0):.1%}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Ham', 'Spam'], [ham_count, spam_count], color=['#2196F3', '#F44336'])
ax.set_ylabel('Count')
ax.set_title('Class distribution: ham vs spam', fontweight='bold')
for i, v in enumerate([ham_count, spam_count]):
    ax.text(i, v + 30, f'{v:,}\n({v/len(raw):.1%})', ha='center', fontsize=10)
plt.tight_layout()
plt.show()


### Step 2C — Read the raw text

**Hint:** Print 5 spam examples and 5 ham examples.
Limit each message to 130 characters using string slicing.


In [ ]:
print('=== SPAM examples ===')
for i, msg in enumerate(raw[raw['spam']==1]['message'].head(5), 1):
    print(f'{i}. {msg[:130]}')
    print()

print('=== HAM examples ===')
for i, msg in enumerate(raw[raw['spam']==0]['message'].head(5), 1):
    print(f'{i}. {msg[:130]}')
    print()


### Step 2D — Message length analysis

**Hint:** Add a column `msg_length = message.str.len()`.
Print average length for spam and ham.
Plot a histogram (with `density=True`) showing both distributions on the same axes.


In [ ]:
raw['msg_length'] = raw['message'].str.len()

print('Average message length:')
print(f'  Ham  : {raw[raw["spam"]==0]["msg_length"].mean():.0f} characters')
print(f'  Spam : {raw[raw["spam"]==1]["msg_length"].mean():.0f} characters')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw[raw['spam']==0]['msg_length'], bins=40, alpha=0.7,
             color='#2196F3', label='Ham', density=True)
axes[0].hist(raw[raw['spam']==1]['msg_length'], bins=40, alpha=0.7,
             color='#F44336', label='Spam', density=True)
axes[0].set_xlabel('Message length (characters)')
axes[0].set_ylabel('Density')
axes[0].set_title('Message length distribution')
axes[0].legend()

axes[1].boxplot(
    [raw[raw['spam']==0]['msg_length'], raw[raw['spam']==1]['msg_length']],
    patch_artist=True,
    boxprops=dict(facecolor='#E3F2FD'),
    medianprops=dict(color='#1565C0', linewidth=2)
)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['Ham', 'Spam'])
axes[1].set_ylabel('Message length (characters)')
axes[1].set_title('Ham vs spam: length comparison')
plt.suptitle('Step 2D — does message length separate the classes?', fontweight='bold')
plt.tight_layout()
plt.show()


---

---
## Step 3 — Feature Engineering

Raw text cannot go directly into a classifier.
Convert it into **binary features** (0 or 1) — the same format as the Hands-On Lab.

### Step 3A — Pre-built starter features
This cell is given to you. Run it and study the patterns before adding your own.


In [ ]:
# This cell is provided — do not modify it
def build_features(df):
    d = df.copy()
    msg = d['message'].str.lower()

    d['has_phone'] = msg.str.contains(
        r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b|call.*\d{4,}|\b0\d{9,}\b',
        regex=True).astype(int)

    d['has_money'] = msg.str.contains(
        r'\b(free|win|winner|prize|cash|\$|claim|reward|won)\b',
        regex=True).astype(int)

    d['has_urgency'] = msg.str.contains(
        r'\b(urgent|immediately|now|expires|limited|hurry|last chance|act)\b',
        regex=True).astype(int)

    d['is_long'] = (d['message'].str.len() > 130).astype(int)
    return d

df = build_features(raw)
starter = ['has_phone', 'has_money', 'has_urgency', 'is_long']
print('Starter features (spam vs ham rate):')
print(f'{"Feature":<16} {"Spam%":>7} {"Ham%":>7} {"Gap":>7}')
print('-' * 42)
for f in starter:
    sp = df[df['spam']==1][f].mean()
    ha = df[df['spam']==0][f].mean()
    print(f'{f:<16} {sp:>7.1%} {ha:>7.1%} {sp-ha:>+7.1%}')


### Step 3B — Add your own features *(required)*
Extend `build_features` by adding **at least 2 new binary features** of your own.

**Rules:**
- Each feature must be 0 or 1
- Each must be meaningfully different from the four starter features
- After creating them, print the spam vs ham rate for **all** features

**Ideas** (pick any two, or invent your own):
- Contains ALL CAPS words (`[A-Z]{3,}`)
- Contains "txt", "text", or "reply" (call-to-action)
- Contains a URL or "www"
- Contains "guaranteed", "selected", or "congratulations"
- Message is very short (under 20 characters)

**Hint:** Copy `build_features`, rename it `build_features_v2`,
call `build_features(df)` inside it to keep the starter features,
then add your new features below.

At the end of the cell, run this line to collect all feature column names:
```python
feature_cols = [c for c in df.columns
                if c not in ['label', 'message', 'spam', 'msg_length']]
```
This variable is used in Steps 4, 5, and 6 — make sure it is defined here.


In [ ]:
def build_features_v2(df):
    d = build_features(df)   # keeps all 4 starter features
    msg = d['message'].str.lower()

    # --- Feature 5: call-to-action keywords (txt / text / reply / send / stop) ---
    d['has_cta'] = msg.str.contains(
        r'\b(txt|text|reply|send|stop)\b',
        regex=True).astype(int)

    # --- Feature 6: ALL-CAPS word of 3+ letters (uses ORIGINAL case) ---
    d['has_caps'] = d['message'].str.contains(
        r'\b[A-Z]{3,}\b',
        regex=True).astype(int)

    # --- Feature 7: UK short code (5-digit number used for premium SMS) ---
    d['has_shortcode'] = msg.str.contains(
        r'(?<!\d)\d{5}(?!\d)',
        regex=True).astype(int)

    return d

df = build_features_v2(raw)

# Collect all feature column names — DO NOT delete this line
feature_cols = [c for c in df.columns
                if c not in ['label', 'message', 'spam', 'msg_length']]

# Print spam vs ham rate for all features
print('All features (spam vs ham rate):')
print(f'{"Feature":<20} {"Spam%":>7} {"Ham%":>7} {"Gap":>7}')
print('-' * 48)
for feat in feature_cols:
    sp = df[df['spam']==1][feat].mean()
    ha = df[df['spam']==0][feat].mean()
    print(f'{feat:<20} {sp:>7.1%} {ha:>7.1%} {sp-ha:>+7.1%}')


### Step 3C — Visualize feature gaps

**Hint:** Create one bar chart per feature (use `plt.subplots(1, n_features, ...)`).
Each bar chart shows two bars: spam rate (red) and ham rate (blue).
This is the same chart style used in the Hands-On Lab Step 0.


In [ ]:
n_features = len(feature_cols)
fig, axes = plt.subplots(1, n_features, figsize=(2.6 * n_features, 3.6), sharey=True)

for ax, feat in zip(axes, feature_cols):
    sp = df[df['spam']==1][feat].mean()
    ha = df[df['spam']==0][feat].mean()
    bars = ax.bar(['Spam', 'Ham'], [sp, ha], color=['#F44336', '#2196F3'])
    for b, v in zip(bars, [sp, ha]):
        ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.0%}',
                ha='center', fontsize=9)
    ax.set_title(f'{feat}\ngap {sp-ha:+.0%}', fontsize=10)
    ax.set_ylim(0, 1.1)

axes[0].set_ylabel('Share of messages with feature = 1')
plt.suptitle('Step 3C — spam vs ham rate per feature', fontweight='bold')
plt.tight_layout()
plt.show()


### Q3 — Feature Engineering *(20 points)*

**Q3a.** Describe the two features you added.
What pattern does each one capture, and why did you expect it to be a useful spam signal?

*Feature 1 — name and explanation:*
> **`has_caps`**: equals 1 if the original (not lowercased) message contains a word of 3+ capital letters, e.g. "WINNER", "URGENT", "FREE", "CASH". Spammers shout in capital letters to grab attention, while friends texting each other rarely do. Result: 70.4% of spam vs 7.2% of ham (gap +63.3%).

*Feature 2 — name and explanation:*
> **`has_cta`**: equals 1 if the message contains a call-to-action word: "txt", "text", "reply", "send" or "stop". Spam SMS almost always asks the user to do something ("Text FA to 87121", "Reply STOP to opt out"), while normal conversation usually does not. Result: 48.9% of spam vs 5.6% of ham (gap +43.3%).
>
> *Bonus feature* — **`has_shortcode`**: equals 1 if the message contains a standalone 5-digit number (a premium-SMS short code such as 87121 or 87575). It is different from `has_phone` because short codes are too short to match the phone-number pattern. Result: 34.0% of spam vs 0.0% of ham, so it almost never fires on ham.

**Q3b.** Which feature has the largest gap between spam and ham?
Why does a large gap matter when a Decision Tree selects its root node?
Connect your answer to the concept of **Information Gain** from Lecture 8.

*Your answer:*
> **`has_caps`** has the largest gap (+63.3%: 70.4% spam vs 7.2% ham), slightly ahead of `is_long` (+62.7%) and `has_phone` (+55.1%). At every node, a Decision Tree tests each feature and picks the one with the highest **Information Gain**, meaning the largest drop in impurity (Gini/entropy) from the parent node to the child nodes. A feature with a large gap splits the data into children that are much "purer" than the parent (one side mostly ham, the other side mostly spam), so it produces the biggest impurity reduction. That is exactly what happened here: the tree chose `has_caps` as its **root node** (Step 5A), and it also has the highest feature importance (0.581 in Step 5B).

**Q3c.** Look at the spam examples from Step 2C.
Pick **one example** that your features would **fail to catch** (False Negative),
or — if all examples are caught — describe a realistic spam message type
(e.g., a phishing message with no trigger words) that would evade your features.
State which specific feature(s) fail and explain why.

*Your answer:*
> **Spam example #2**: *"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv"*. Following the tree, it goes `has_caps = 0 → has_phone = 0 → has_shortcode = 0 → has_money = 0 → Ham`, so it is a **False Negative**. `has_money` fails because "free" is glued into "FreeMsg", so the word-boundary regex `\bfree\b` does not match. `has_caps` fails because "XxX" is mixed-case, not all caps. `has_phone` and `has_shortcode` fail because the message contains no phone number or short code at all (it ends with "std chgs to send, £1.50 to rcv"). It is written like a casual, flirty personal message, which is exactly what ham looks like, so keyword-based features cannot separate it.


---
## Step 4 — Split the Data

### Step 4A — Train/Test split

**Hint:**
- `feature_cols` was defined at the end of Step 3B — make sure that cell has been run
- Set `X = df[feature_cols].values` and `y = df['spam'].values`
- Use `train_test_split` with `test_size=0.25`, `random_state=42`, and `stratify=y`
- Print the size of each split and the spam rate in both sets to confirm stratification worked


In [ ]:
X = df[feature_cols].values
y = df['spam'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print(f'Features used : {feature_cols}')
print(f'Training set  : {len(X_train):,} messages   spam rate = {y_train.mean():.1%}')
print(f'Test set      : {len(X_test):,} messages   spam rate = {y_test.mean():.1%}')
print(f'Full dataset  : {len(X):,} messages   spam rate = {y.mean():.1%}')


### Q4 — Data Split *(10 points)*

**Q4a. (5 points)** Why do we pass `stratify=y`?
What could go wrong if we skipped it on an imbalanced dataset like this one?

*Your answer:*
> `stratify=y` makes the train and test sets keep the same class ratio as the full dataset. Step 4A confirms this: 13.4% spam in both the training set and the test set. Without it, the random split could put, for example, 10% of the spam in the test set and 15% in the training set. Since spam is only 13.4% of the data (747 messages), a small shift changes the results a lot: the model would learn from a different class balance than it is tested on, and precision, recall and F1 on the test set would be unreliable and could change a lot just by changing the random seed.

**Q4b. (5 points)** We use a 75/25 train/test split.
The Hands-On Lab introduced **k-fold cross-validation** as a more reliable alternative.
Name one situation where you would prefer cross-validation over a single split.

*Your answer:*
> When **comparing two models whose scores are close** (like our Decision Tree and Naive Bayes, whose test F1 differ by only about 0.02), or when **tuning a hyperparameter** like `max_depth`. A single split gives one number that depends on which messages ended up in the test set, so the "winner" could just be luck. 5-fold cross-validation trains and tests 5 times so every message is used for testing once, and the average (and its spread) tells us whether the difference is real. It is also preferred when the dataset is small, because no data is permanently "wasted" on the test set.


---
## Step 5 — Train the Models （Each code 5 pts）

### Step 5A — Decision Tree

**Hint:**
- Use `DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')`
- Fit on the training set
- Print the number of leaves with `.get_n_leaves()`
- Visualize the tree with `plot_tree(..., feature_names=..., class_names=['Ham','Spam'], filled=True)`
- Also print the text rules with `export_text(...)`


In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

print(f'Number of leaves: {dt.get_n_leaves()}')
print(f'Tree depth      : {dt.get_depth()}')

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(dt, feature_names=feature_cols, class_names=['Ham', 'Spam'],
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.title('Step 5A — Decision Tree (max_depth=4, class_weight=balanced)', fontweight='bold')
plt.show()

print(export_text(dt, feature_names=feature_cols))


### Step 5B — Feature Importance

**Hint:**
- Get `.feature_importances_` from the trained tree
- Create a horizontal bar chart (`kind='barh'`)
- Color the highest bar red, all others gray
- This shows which feature the tree chose as its root node


In [ ]:
importances = pd.Series(dt.feature_importances_, index=feature_cols).sort_values()
colors = ['#F44336' if v == importances.max() else '#9E9E9E' for v in importances]

ax = importances.plot(kind='barh', color=colors, figsize=(7, 4))
ax.set_xlabel('Feature importance (total Gini reduction)')
ax.set_title('Step 5B — Decision Tree feature importance', fontweight='bold')
for i, v in enumerate(importances):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Root node feature: {feature_cols[dt.tree_.feature[0]]}')
print(f'Most important   : {importances.idxmax()} ({importances.max():.3f})')


### Step 5C — Effect of Tree Depth

**Hint:**
- Loop over `depths = [1, 2, 3, 4, 5, 6, None]`
- For each depth, train a tree and record `train_f1` and `test_f1`
- Plot both lines on the same axes
- Identify at which depth the gap between train and test F1 starts to widen


In [ ]:
depths = [1, 2, 3, 4, 5, 6, None]
train_f1, test_f1 = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42, class_weight='balanced')
    m.fit(X_train, y_train)
    train_f1.append(f1_score(y_train, m.predict(X_train)))
    test_f1.append(f1_score(y_test, m.predict(X_test)))

labels = [str(d) for d in depths]
print(f'{"Depth":<6} {"Train F1":>9} {"Test F1":>9} {"Gap":>8}')
for lab, tr, te in zip(labels, train_f1, test_f1):
    print(f'{lab:<6} {tr:>9.3f} {te:>9.3f} {tr-te:>+8.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(labels, train_f1, marker='o', color='#2196F3', label='Train F1')
ax.plot(labels, test_f1, marker='o', color='#F44336', label='Test F1')
ax.set_xlabel('max_depth')
ax.set_ylabel('F1-Score (spam class)')
ax.set_title('Step 5C — effect of tree depth', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('Observation: the train/test gap is largest for very shallow trees (underfitting, depth 1-3),')
print('stays near zero at depth 4-5, and opens up again from depth 6 (train 0.827 vs test 0.805).')
print('With only 7 binary features (2^7 = 128 possible inputs) the tree has little room to overfit,')
print('so depth 4-5 is the sweet spot; max_depth=4 used in Step 5A is a reasonable choice.')


### Step 5D — Naive Bayes

**Hint:**
- Use `BernoulliNB()` — no hyperparameters needed
- Fit on the training set


In [ ]:
nb = BernoulliNB()
nb.fit(X_train, y_train)

print('Naive Bayes trained.')
print(f'Class priors (ham, spam): {np.exp(nb.class_log_prior_).round(3)}')
print()
print('P(feature = 1 | class):')
probs = pd.DataFrame(np.exp(nb.feature_log_prob_).T, index=feature_cols,
                     columns=['Ham', 'Spam']).round(3)
print(probs)


---
## Step 6 — Evaluate *(Each code block 4 points)*

### Step 6A — Confusion Matrices

**Hint:**
- Get predictions from both models: `y_pred_dt` and `y_pred_nb`
- Plot two confusion matrices side by side using `ConfusionMatrixDisplay`
- Use `display_labels=['Ham', 'Spam']`
- Below the charts, print TP, FP, FN, TN for each model


In [ ]:
y_pred_dt = dt.predict(X_test)
y_pred_nb = nb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, pred, name, cmap in [(axes[0], y_pred_dt, 'Decision Tree', 'Blues'),
                             (axes[1], y_pred_nb, 'Naive Bayes', 'Oranges')]:
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                           display_labels=['Ham', 'Spam']).plot(ax=ax, cmap=cmap, colorbar=False)
    ax.set_title(name, fontweight='bold')
plt.suptitle('Step 6A — confusion matrices (test set)', fontweight='bold')
plt.tight_layout()
plt.show()

for name, pred in [('Decision Tree', y_pred_dt), ('Naive Bayes', y_pred_nb)]:
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    print(f'{name:<14} TP={tp:<4} FP={fp:<4} FN={fn:<4} TN={tn}')


### Step 6B — Metrics Comparison

**Hint:**
- Build a DataFrame with columns: Model, Accuracy, Precision, Recall, F1-Score
- One row per model
- Also create a grouped bar chart comparing all four metrics side by side


In [ ]:
rows = []
for name, pred in [('Decision Tree', y_pred_dt), ('Naive Bayes', y_pred_nb)]:
    rows.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall':    recall_score(y_test, pred),
        'F1-Score':  f1_score(y_test, pred),
    })
results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, color in enumerate(['#2196F3', '#FF9800']):
    row = results.iloc[i]
    vals = [row[m] for m in metrics]
    bars = ax.bar(x + (i - 0.5) * w, vals, w, label=row['Model'], color=color)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Step 6B — metrics comparison (test set)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


### Step 6C — Cross-Validation

**Hint:**
- Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- Run `cross_val_score` with `scoring='f1'` for both models
- Print the average F1 across 5 folds for each model


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_dt = cross_val_score(
    DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced'),
    X, y, cv=skf, scoring='f1')
cv_nb = cross_val_score(BernoulliNB(), X, y, cv=skf, scoring='f1')

print('5-fold stratified cross-validation (F1, spam class):')
print(f'  Decision Tree: folds = {np.round(cv_dt, 3)}  mean = {cv_dt.mean():.3f}  std = {cv_dt.std():.3f}')
print(f'  Naive Bayes  : folds = {np.round(cv_nb, 3)}  mean = {cv_nb.mean():.3f}  std = {cv_nb.std():.3f}')


### Q6 — Evaluation & Analysis *(25 points)*

> **Points breakdown:**
> - Step 6A code (runs correctly): 4 pts
> - Step 6B code (runs correctly): 4 pts
> - Step 6C code (runs correctly): 4 pts
> - Q6a written: 3 pts
> - Q6b written: 3 pts
> - Q6c written: 4 pts
> - Q6d written: 3 pts
> - **Total: 25 pts**

---

**Q6a. Read the confusion matrices (Step 6A) — *(3 points)***

For each model, state:
- How many spam messages were correctly caught (True Positives)?
- How many spam messages slipped through (False Negatives)?
- How many legitimate messages were wrongly blocked (False Positives)?

*Decision Tree:*
> The test set has 187 spam and 1,206 ham messages. **TP = 169** spam caught, **FN = 18** spam slipped through, **FP = 34** legitimate messages wrongly blocked (TN = 1,172).

*Naive Bayes:*
> **TP = 161** spam caught, **FN = 26** spam slipped through, **FP = 16** legitimate messages wrongly blocked (TN = 1,190).

**Q6b. Apply your metric from Q1b — *(3 points)***

Which model wins on the metric you chose in Step 1?
By how much? Does the cross-validation result (Step 6C) confirm this?

*Your answer:*
> My primary metric is **Precision**, and **Naive Bayes wins**: 0.910 vs 0.833 for the Decision Tree, about **+7.7 percentage points**. In practice, Naive Bayes blocks 16 real messages instead of 34, so it cuts False Positives by more than half. The Decision Tree has higher recall (0.904 vs 0.861) because `class_weight='balanced'` pushes it to flag more spam, but that is the trade-off we decided not to make in Q1. Cross-validation uses F1, not precision, but it points in the same direction: Naive Bayes averages **0.890** F1 vs **0.872** for the Decision Tree across 5 folds, and it is also more stable (its fold scores vary less). So the CV result confirms that Naive Bayes is the better model overall, not just on this one split.

**Q6c. Recommendation memo — *(4 points)***

Imagine you are presenting to a product manager at a mobile carrier.
Write 4–5 sentences that:
1. State which model you recommend
2. Justify the choice with one specific number from your results
3. Acknowledge one weakness of your recommended model
4. Propose one concrete improvement

*Your memo:*
> I recommend deploying the **Naive Bayes** classifier as our SMS spam filter. When it blocks a message, it is right **91% of the time** (precision 0.910), and on our test set it wrongly blocked only 16 out of 1,206 real messages, less than half of the Decision Tree's 34. Its main weakness is that it lets more spam through: it misses about 14% of spam (26 of 187 messages, recall 0.861), especially casual-sounding spam with no keywords, numbers or capital letters. To improve it, I propose replacing our 7 hand-made yes/no features with word-level features (bag-of-words or TF-IDF over the whole vocabulary) so the model can learn spam patterns we did not think of, and then tuning the decision threshold so precision stays high while recall goes up.

**Q6d. Generalization — *(3 points)***

The SMS Spam Collection was collected in the early 2010s.
Modern spam often uses professional language and legitimate-looking URLs.
Identify **two features you built** that would likely fail on modern spam, and explain why.

*Your answer:*
> 1. **`has_caps`**: modern spam imitates real companies ("USPS: Your package is on hold. Update your address at …"), so it uses normal, professional sentence case instead of "WINNER!!" or "FREE CASH". It would score 0 and look like ham, and since it is our root node, the whole tree would be fooled.
> 2. **`has_shortcode`** (and `has_cta`): 2010s spam asked users to text a 5-digit premium short code ("txt CSH11 to 87575"). Modern smishing sends the user to a **link** instead, often a URL shortener or a look-alike domain, and comes from ordinary 10-digit numbers or email-to-SMS gateways. These messages contain no short code and no "txt/reply/send" words, so both features would be 0. A feature that checks links (URL present, shortened URL, domain that doesn't match the brand named in the message) would be needed instead.
